# 07 · Predictions & Validation
Phase 7: Assemble final predictions, backtest vs actuals, generate SUMMARY.md.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from src.data_loader import load_all, CONFEDERATION_MAP
from src.dixon_coles import DixonColes
from src.elo_calculator import build_elo_ratings
from src.feature_engineer import RollingStatsEngine
from src.models import XGBOutcomeModel, EnsemblePredictor, CornersModel, YellowCardsModel
from src.evaluator import Backtest, PredictionAssembler, generate_summary_report
np.random.seed(42)

data = load_all(verbose=False)
results_full = data['results_full']
fixtures = data['group_fixtures']
ko_slots = data['knockout_slots']
rankings = data['fifa_rankings']
registry = data['team_registry']

print("Loading processed data...")
wc_feat = pd.read_csv('../data/processed/wc2026_features.csv')
winner_probs = pd.read_csv('../data/processed/tournament_winner_probs.csv')
ko_preds_sim = pd.read_csv('../data/processed/knockout_predictions.csv')
group_probs  = pd.read_csv('../data/processed/group_stage_probabilities.csv')
print("All processed files loaded ✅")


## 7.1 Rebuild Models for Prediction Assembly

In [ ]:
# Rebuild Elo
elo_path = Path('../data/processed/team_elo_ratings.csv')
if elo_path.exists():
    elo_df = pd.read_csv(elo_path)
    elo_ratings = dict(zip(elo_df['team'], elo_df['elo_rating']))
else:
    _, elo_ratings = build_elo_ratings(results_full, save=True)

# Rebuild Dixon-Coles
train_final = results_full[results_full['date'].dt.year >= 2015].copy()
dc_model = DixonColes(xi=0.003)
dc_model.fit(train_final)
print("Dixon-Coles ready")

# Load XGBoost
try:
    xgb_model = XGBOutcomeModel.load()
    print("XGBoost loaded from disk")
except Exception:
    print("XGBoost not found — fitting with defaults...")
    feat_df = pd.read_csv('../data/processed/match_features.csv', parse_dates=['date'])
    train_xgb = feat_df[feat_df['date'].dt.year >= 2010].dropna(subset=['outcome'])
    xgb_model = XGBOutcomeModel()
    xgb_model.fit(train_xgb)
    xgb_model.save()

ensemble = EnsemblePredictor(dc_weight=0.6, xgb_weight=0.4)
corners_model = CornersModel()
yellows_model = YellowCardsModel()

assembler = PredictionAssembler(dc_model, xgb_model, ensemble, corners_model, yellows_model)
print("All models ready ✅")


## 7.2 Assemble Group Stage Predictions

In [ ]:
print("Assembling group stage predictions...")
group_preds = assembler.assemble_group_predictions(wc_feat)

print(f"Generated {len(group_preds)} match predictions")
print("\nSample predictions:")
cols = ['match_id','group','home_team','away_team','predicted_home_goals',
        'predicted_away_goals','predicted_outcome','p_home_win','p_draw',
        'p_away_win','corners','yellow_cards']
print(group_preds[cols].head(12).to_string(index=False))


## 7.3 Backtest Against Played Group Stage Matches

In [ ]:
# Filter results to 2026 WC group stage dates
wc_2026_actual = results_full[
    (results_full['is_wc']) &
    (results_full['date'] >= pd.Timestamp('2026-06-11')) &
    (results_full['date'] <= pd.Timestamp('2026-06-28'))
].copy()

print(f"Actual WC 2026 results found in results.csv: {len(wc_2026_actual)} matches")

if len(wc_2026_actual) > 0:
    backtest = Backtest(group_preds, wc_2026_actual)
    backtest.merge()
    model_metrics = backtest.compute_metrics()
    naive_metrics = backtest.compute_naive_metrics()
    print(backtest.report())
else:
    print("⚠️  No 2026 WC results available in results.csv yet.")
    print("   (The dataset may not have been updated post June 11, 2026.)")
    print("   To backtest manually: add actual scores to group_preds and compare.")
    model_metrics = {'n_matches': 0, 'note': 'No actuals available in dataset'}
    naive_metrics = {}


## 7.4 Assemble Knockout Predictions

In [ ]:
sim_results_dict = {
    'winner_probs':   winner_probs,
    'ko_predictions': ko_preds_sim,
    'group_probs':    group_probs,
}
ko_preds_final = assembler.assemble_knockout_predictions(
    sim_results_dict, wc_feat, ko_slots
)

print("Knockout Predictions Summary:")
for rnd in ['Round of 32','Round of 16','Quarter-final','Semi-final','Final']:
    rnd_df = ko_preds_final[ko_preds_final['round'] == rnd]
    print(f"\n{rnd}:")
    for _, r in rnd_df.iterrows():
        winner = r.get('match_winner', '?')
        print(f"  {r.get('predicted_home','TBD'):<22} vs {r.get('predicted_away','TBD'):<22}  → {winner}")


## 7.5 Upset Alert Analysis

In [ ]:
upsets = []
for _, row in group_preds.iterrows():
    rh = row.get('fifa_rank_home', 999) if not pd.isna(row.get('fifa_rank_home', np.nan)) else 999
    ra = row.get('fifa_rank_away', 999) if not pd.isna(row.get('fifa_rank_away', np.nan)) else 999
    if rh < ra:  # home is favourite
        upset_prob = row.get('p_away_win', 0)
        upset_team, fav_team = row['away_team'], row['home_team']
    else:
        upset_prob = row.get('p_home_win', 0)
        upset_team, fav_team = row['home_team'], row['away_team']
    if upset_prob > 0.40:
        upsets.append({
            'match': f"{row['home_team']} vs {row['away_team']}",
            'group': row['group'],
            'underdog': upset_team, 'favourite': fav_team,
            'upset_prob': upset_prob,
        })

upsets = sorted(upsets, key=lambda x: -x['upset_prob'])
print("⚡ TOP UPSET ALERTS (>40% probability for lower-ranked team):")
print("=" * 60)
for u in upsets[:10]:
    print(f"  Group {u['group']}: {u['underdog']} ({u['upset_prob']:.0%}) to upset {u['favourite']}")


## 7.6 Generate SUMMARY.md Report

In [ ]:
summary = generate_summary_report(
    winner_probs=winner_probs,
    ko_preds=ko_preds_final,
    group_preds=group_preds,
    group_stage_probs=group_probs,
    model_metrics=model_metrics,
    naive_metrics=naive_metrics,
    output_path='../outputs/predictions/SUMMARY.md'
)
print("SUMMARY.md generated!")
print("\n--- Preview ---")
print(summary[:2000])


## ✅ Pipeline Complete!

All 7 phases executed. Key outputs:
- `outputs/predictions/group_stage_predictions.csv`
- `outputs/predictions/knockout_predictions.csv`
- `outputs/predictions/SUMMARY.md`
- `outputs/plots/*.png`
- `outputs/model_artifacts/`